In [2]:
import wrds
import pandas as pd
import numpy as np
from typing import Tuple, List, Dict
from pathlib import Path

def g_cmp(d: wrds.Connection, g: str) -> pd.DataFrame:
    """Retrieves global fundamental data from Compustat using GVKEY."""
    q = f"SELECT datadate, at, lt, nit, revt FROM comp.g_funda WHERE gvkey = '{g}' AND indfmt = 'INDL' AND datafmt = 'STD' AND popsrc = 'I' AND consol = 'C' ORDER BY datadate ASC"
    return d.raw_sql(q, date_cols=['datadate'])

def g_ibs_int(d: wrds.Connection, t: str) -> pd.DataFrame:
    """Retrieves International consensus estimates from LSEG IBES."""
    q = f"SELECT statpers, fpedats, meanest, numest FROM ibes.statsumu_epsint WHERE ticker = '{t}' ORDER BY statpers ASC"
    return d.raw_sql(q, date_cols=['statpers', 'fpedats'])

def g_crs(d: wrds.Connection, p: int, s: str, e: str) -> pd.DataFrame:
    """Retrieves daily stock data from CRSP Version 2 using PERMNO."""
    q = f"SELECT dlycaldt, dlyret, dlyvol FROM crsp.dsf_v2 WHERE permno = {p} AND dlycaldt >= '{s}' AND dlycaldt <= '{e}' ORDER BY dlycaldt ASC"
    return d.raw_sql(q, date_cols=['dlycaldt'])

def g_evt(d: wrds.Connection, c: str) -> pd.DataFrame:
    """Retrieves key developments from Capital IQ using CompanyID."""
    q = f"SELECT a.announcedate, a.keydeveventtypeid, b.headline FROM ciq.wrds_keydev a JOIN ciq.ciqkeydev b ON a.keydevid = b.keydevid WHERE a.companyid = {c} ORDER BY a.announcedate ASC"
    return d.raw_sql(q, date_cols=['announcedate'])

def g_ff(d: wrds.Connection, s: str, e: str) -> pd.DataFrame:
    """Retrieves Fama-French 5-Factor daily data."""
    q = f"SELECT date, mktrf, smb, hml, rmw, cma, rf FROM ff.fivefactors_daily WHERE date >= '{s}' AND date <= '{e}' ORDER BY date ASC"
    return d.raw_sql(q, date_cols=['date']).rename(columns={'date': 'ff_date'})

def g_shv(d: wrds.Connection, t: str, s: str, e: str) -> pd.DataFrame:
    """Retrieves short interest data."""
    q = f"SELECT date, shortint FROM comp.sec_shortint WHERE tic = '{t}' AND date >= '{s}' AND date <= '{e}' ORDER BY date ASC"
    try:
        return d.raw_sql(q, date_cols=['date']).rename(columns={'date': 'shv_date'})
    except Exception:
        return pd.DataFrame(columns=['shv_date', 'shortint'])

def b_pipe(c: pd.DataFrame, i: pd.DataFrame, r: pd.DataFrame, e: pd.DataFrame, ff: pd.DataFrame, shv: pd.DataFrame) -> pd.DataFrame:
    """Builds an aligned, forward-filled feature matrix."""
    r = r.dropna(subset=['dlycaldt']).sort_values('dlycaldt')
    c = c.dropna(subset=['datadate']).sort_values('datadate')
    i = i.dropna(subset=['statpers']).sort_values('statpers')
    ff = ff.dropna(subset=['ff_date']).sort_values('ff_date')
    shv = shv.dropna(subset=['shv_date']).sort_values('shv_date')

    m = pd.merge_asof(r, c, left_on='dlycaldt', right_on='datadate', direction='backward')
    m = pd.merge_asof(m, i, left_on='dlycaldt', right_on='statpers', direction='backward')
    m = pd.merge_asof(m, ff, left_on='dlycaldt', right_on='ff_date', direction='backward')

    if not shv.empty:
        m = pd.merge_asof(m, shv, left_on='dlycaldt', right_on='shv_date', direction='backward')

    e_g = e.groupby('announcedate').agg({'keydeveventtypeid': list, 'headline': list}).reset_index()
    f = pd.merge(m, e_g, left_on='dlycaldt', right_on='announcedate', how='left')

    d_cols = ['datadate', 'statpers', 'ff_date', 'shv_date', 'announcedate']
    f.drop(columns=[col for col in d_cols if col in f.columns], inplace=True)

    return f

In [3]:
def g_sch(d: wrds.Connection, k: str) -> pd.DataFrame:
    """Lists schema.table pairs whose schema name matches a keyword."""
    q = "SELECT table_schema, table_name FROM information_schema.tables WHERE table_schema LIKE %(k)s ORDER BY table_schema, table_name"
    return d.raw_sql(q, params={'k': f'%{k}%'})

def g_col(d: wrds.Connection, schema: str, table: str) -> pd.DataFrame:
    """Lists columns and their data types for a given schema.table."""
    q = "SELECT column_name, data_type FROM information_schema.columns WHERE table_schema = %(s)s AND table_name = %(t)s ORDER BY ordinal_position"
    return d.raw_sql(q, params={'s': schema, 't': table})

In [11]:
usr = "zackienzle1"
gvkey = "012114"
permno = 69032
ibes_tic = "BHP" # International database
ciq_id = "20743"
us_tic = "BHP" # For short interest
st = "2014-01-01"
ed = "2024-01-01"

db = wrds.Connection(wrds_username=usr)

Loading library list...
Done


In [5]:
# for kw in ['comp', 'ibes', 'crsp', 'ciq', 'ff', 'iss', 'boardex', 'betasuite', 'wrdsapps']:
#     print(kw)
#     print(g_sch(db, kw).to_string())

# targets = [
#     ('comp',      'g_security'),
#     ('comp',      'g_funda'),
#     ('comp',      'security'),
#     ('comp',      'funda'),
#     ('comp',      'sec_shortint'),
#     ('ibes',      'idsum'),
#     ('ibes',      'statsumu_epsus'),
#     ('crsp',      'dsf_v2'),
#     ('ciq',       'wrds_keydev'),
#     ('ciq',       'ciqkeydev'),
#     ('ciq',       'wrds_ticker'),
#     ('ff',        'fivefactors_daily'),
#     ('betasuite', 'beta_daily'),
#     ('wrdsapps',  'factors_daily'),
# ]
# for s, t in targets:
#     print(f"{s}.{t}")
#     print(g_col(db, s, t).to_string())

# print("BHP gvkey")
# print(db.raw_sql("SELECT gvkey, tic, dldtei, sedol, isin FROM comp.g_security WHERE tic = %(t)s", params={'t': 'BHP'}))
# print("BHP ibes")
# print(db.raw_sql("SELECT ticker, cusip, cname, sdates FROM ibes.idsum WHERE ticker = %(t)s ORDER BY sdates DESC LIMIT 5", params={'t': 'BHP'}))

# db.close()

In [6]:
df_cmp = g_cmp(db, gvkey)
df_ibs = g_ibs_int(db, ibes_tic)
df_crs = g_crs(db, permno, st, ed)
df_evt = g_evt(db, ciq_id)
df_ff = g_ff(db, st, ed)
df_shv = g_shv(db, us_tic, st, ed)

db.close()

df_main = b_pipe(df_cmp, df_ibs, df_crs, df_evt, df_ff, df_shv)

In [7]:
df_main

,dlycaldt,dlyret,dlyvol,at,lt,nit,revt,fpedats,meanest,numest,mktrf,smb,hml,rmw,cma,rf,keydeveventtypeid,headline
0,2014-01-02,-0.010523,9843400.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,-0.0088,-0.0025,0.0017,-0.0032,0.0011,0.0,NaN,NaN
1,2014-01-03,0.015469,7811100.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,0.0003,0.0041,0.0003,-0.0035,0.0015,0.0,NaN,NaN
2,2014-01-06,0.003491,9031700.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,-0.0034,-0.0054,0.003,-0.0031,0.0005,0.0,NaN,NaN
3,2014-01-07,-0.003163,9246100.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,0.0068,0.0032,-0.0038,-0.0008,-0.003,0.0,NaN,NaN
4,2014-01-08,0.001269,8930200.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,0.0004,-0.0001,-0.001,-0.0047,-0.0007,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2511,2023-12-22,-0.002375,5800929.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,0.002,0.0061,0.001,-0.0064,0.002,0.0002,NaN,NaN
2512,2023-12-26,0.004653,2726286.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,0.0048,0.0082,0.0044,-0.0032,-0.0015,0.0002,NaN,NaN
2513,2023-12-27,0.008832,4073222.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,0.0016,0.0016,0.0011,-0.0032,-0.0014,0.0002,NaN,NaN
2514,2023-12-28,-0.000214,4089529.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,-0.0001,-0.0039,0.0003,-0.0031,0.0016,0.0002,NaN,NaN


In [8]:
def s_csv(d_m: Dict[str, pd.DataFrame], d_p: str = "../data") -> None:
    """Saves multiple DataFrames to CSV efficiently."""
    p = Path(d_p)
    p.mkdir(parents=True, exist_ok=True)
    for k, v in d_m.items():
        if not v.empty:
            v.to_csv(p / f"{k}.csv", index=False, chunksize=100000)

In [9]:
d_out = {
    "bhp_cmp": df_cmp,
    "bhp_ibs": df_ibs,
    "bhp_crs": df_crs,
    "bhp_evt": df_evt,
    "bhp_ff": df_ff,
    "bhp_shv": df_shv,
    "bhp_main": df_main
}

s_csv(d_out)
print(f"Pipeline executed. Main matrix shape: {df_main.shape}")

Pipeline executed. Main matrix shape: (2516, 18)
